## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login
```

or

```bash
az login --use-device-code
```

# 🔌 Azure AI Agent with Hosted MCP Tools

This notebook demonstrates integration of Azure AI Agents with hosted Model Context Protocol (MCP) servers using `FoundryAgent`, including user approval workflows for function call security.

## Features Covered:
- Setting up Azure AI Agents with Hosted MCP tools (configured server-side)
- Connecting to external MCP servers for enhanced capabilities
- Implementing user approval workflows for secure function calls
- Thread-based conversation management
- Querying Microsoft Learn documentation via MCP

### ⚠️ Important Note ⚠️
> **MCP (Model Context Protocol) allows agents to access external tools and services. User approval workflows ensure secure function execution.**

## Prerequisites

Before running this notebook, ensure you have:

1. **Azure AI Project**: Access to an Microsoft Foundry project with deployed models
2. **Authentication**: Azure CLI installed and authenticated (`az login --use-device-code`)
3. **Environment Variables**: Set up your `.env` file with:
   - `AI_FOUNDRY_PROJECT_ENDPOINT`
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME`
4. **Dependencies**: Required agent-framework packages installed

If you need to use a different tenant:
```bash
az login --tenant <tenant-id>
```

## Import Libraries

Import the required libraries using the `FoundryAgent` API:

In [ ]:
import os
from pathlib import Path
from typing import Any, Mapping, Sequence

from agent_framework import AgentProtocol, AgentResponse, AgentThread, ChatMessage
from agent_framework._tools import FunctionInvocationLayer
from agent_framework._middleware import ChatMiddlewareLayer
from agent_framework._types import Message
from agent_framework.foundry import FoundryAgent, RawFoundryAgentChatClient
from agent_framework.observability import ChatTelemetryLayer
from azure.identity import AzureCliCredential
from dotenv import load_dotenv


class FoundryAgentChatClient(
    FunctionInvocationLayer,
    ChatMiddlewareLayer,
    ChatTelemetryLayer,
    RawFoundryAgentChatClient,
):
    """Workaround for rc6: strip tool schemas from API request."""

    async def _prepare_options(
        self,
        messages: Sequence[Message],
        options: Mapping[str, Any],
        **kwargs: Any,
    ) -> dict[str, Any]:
        run_options = await super()._prepare_options(messages, options, **kwargs)
        run_options.pop("tools", None)
        run_options.pop("tool_choice", None)
        run_options.pop("parallel_tool_calls", None)
        return run_options


# Load environment variables
notebook_path = Path().absolute()
load_dotenv('../../.env')

# Verify environment setup
endpoint = os.getenv('AI_FOUNDRY_PROJECT_ENDPOINT')
model = os.getenv('AZURE_AI_MODEL_DEPLOYMENT_NAME')

print("🔧 Environment Configuration:")
print(f"✅ Project Endpoint: {endpoint[:50]}..." if endpoint else "❌ AI_FOUNDRY_PROJECT_ENDPOINT not set")
print(f"✅ Model Deployment: {model}" if model else "❌ AZURE_AI_MODEL_DEPLOYMENT_NAME not set")

## Define User Approval Handler 🔐

The approval handler implements a security workflow that requires user confirmation before executing MCP tool calls. This is important for:
- **Security**: Prevents unauthorized function execution
- **Transparency**: Users can see what actions the agent wants to perform
- **Control**: Users maintain control over sensitive operations

In [ ]:
async def handle_approvals_without_thread(query: str, agent: "AgentProtocol") -> AgentResponse:
    """When we don't have a thread, we need to ensure we return with the input,
    approval request and approval."""
    result = await agent.run(query, store=False)
    
    while len(result.user_input_requests) > 0:
        new_inputs: list[Any] = [query]
        
        for user_input_needed in result.user_input_requests:
            print(
                f"🔐 User Input Request for function from {agent.name}: "
                f"{user_input_needed.function_call.name}"
                f" with arguments: {user_input_needed.function_call.arguments}"
            )
            new_inputs.append(ChatMessage(role="assistant", contents=[user_input_needed]))
            user_approval = input("Approve function call? (y/n): ")
            new_inputs.append(
                ChatMessage(role="user", contents=[user_input_needed.create_response(user_approval.lower() == "y")])
            )
        
        result = await agent.run(new_inputs, store=False)
    
    return result


async def handle_approvals_with_thread(
    query: str, 
    agent: "AgentProtocol", 
    thread: "AgentThread"
) -> AgentResponse:
    """Here we let the thread deal with the previous responses, and we just rerun with the approval."""
    result = await agent.run(query, thread=thread)
    
    while len(result.user_input_requests) > 0:
        new_input: list[Any] = []
        
        for user_input_needed in result.user_input_requests:
            print(
                f"🔐 User Input Request for function from {agent.name}: "
                f"{user_input_needed.function_call.name}"
                f" with arguments: {user_input_needed.function_call.arguments}"
            )
            user_approval = input("Approve function call? (y/n): ")
            new_input.append(
                ChatMessage(
                    role="user",
                    contents=[user_input_needed.create_response(user_approval.lower() == "y")],
                )
            )
        
        result = await agent.run(new_input, thread=thread)
    
    return result

## Create Agent with Hosted MCP Tool 🔌

MCP (Model Context Protocol) tools are configured server-side on the Foundry agent. The agent connects to external MCP servers that provide additional capabilities. In this example, the agent connects to Microsoft Learn's MCP endpoint to answer documentation questions.

**Key Concepts:**
- MCP tools are configured on the agent in Azure AI Foundry (not passed client-side)
- Approval modes (never require / always require) are configured server-side
- The `FoundryAgentChatClient` workaround strips tool schemas since tools are managed server-side

In [ ]:
async def run_hosted_mcp_without_approval() -> None:
    """Example showing MCP Tools without approval."""
    print("=== 🔌 MCP without approvals ===")
    
    # Create agent — MCP tools are configured server-side on the Foundry agent
    agent = FoundryAgent(
        agent_name="my-learn-docs-agent",
        project_endpoint=endpoint,
        credential=AzureCliCredential(),
        instructions="You are a helpful assistant that can help with Microsoft documentation questions.",
        client_type=FoundryAgentChatClient,
    )
    
    print(f"✅ Created agent: {agent.name}")
    print(f"🔌 MCP Tool: Microsoft Learn MCP (configured server-side)\n")
    
    query = "How to create an Azure storage account using az cli?"
    print(f"🤔 User: {query}")
    result = await agent.run(query)
    print(f"📚 {agent.name}: {result}\n")

## Execute Without Approval 🚀

Run the example without approval mode - the agent will execute MCP tool calls automatically:

In [ ]:
# Run without approval mode
await run_hosted_mcp_without_approval()

## MCP with Approval Mode 🔐

For scenarios where you want user approval for function calls (recommended for security):

> ⚠️ **Note:** The approval workflow with `approval_mode="always_require"` requires interactive user input which does not work reliably in Jupyter notebooks due to the asynchronous nature of the execution. 
>


In [ ]:
async def run_hosted_mcp_with_approval() -> None:
    """Example showing MCP Tools with approvals (without thread)."""
    print("=== 🔐 MCP with approvals ===")
    
    # Create agent — MCP tools are configured server-side on the Foundry agent
    agent = FoundryAgent(
        agent_name="my-api-specs-agent",
        project_endpoint=endpoint,
        credential=AzureCliCredential(),
        instructions="You are a helpful agent that can use MCP tools to assist users.",
        client_type=FoundryAgentChatClient,
    )
    
    print(f"✅ Created agent: {agent.name}")
    print(f"🔌 MCP Tool: api-specs (configured server-side with approval)\n")
    
    query = "Please summarize the Azure REST API specifications Readme"
    print(f"🤔 User: {query}")
    result = await handle_approvals_without_thread(query, agent)
    print(f"📚 {agent.name}: {result}\n")

# ⚠️ This will NOT work in notebook - run as a script instead
# await run_hosted_mcp_with_approval()

## Key Takeaways 📚

### Hosted MCP Tools

```python
from agent_framework.foundry import FoundryAgent

# MCP tools are configured server-side on the Foundry agent
agent = FoundryAgent(
    agent_name="my-mcp-agent",
    project_endpoint=endpoint,
    credential=AzureCliCredential(),
    instructions="...",
    client_type=FoundryAgentChatClient,
)
```

### User Approval Workflow

1. **Request Detection**: Check `result.user_input_requests` for pending approvals
2. **User Prompt**: Display function name and arguments to user
3. **Approval Response**: Use `user_input_needed.create_response(True/False)`
4. **Re-run Agent**: Continue execution with approval responses

### Benefits of MCP Integration

| Feature | Benefit |
|---------|----------|
| External Services | Access documentation, APIs, and tools |
| Security | User approval workflow for sensitive operations |
| Flexibility | Connect to any MCP-compatible server |
| Context | Thread-based conversation management |

### Approval Modes

Approval modes are configured server-side on the Foundry agent:

| Mode | Description |
|------|-------------|
| Never require | Tool calls execute automatically without user approval |
| Always require | Every tool call requires explicit user approval |

### Available MCP Endpoints

- **Microsoft Learn**: `https://learn.microsoft.com/api/mcp` - Documentation search and retrieval
- **Azure REST API Specs**: `https://gitmcp.io/Azure/azure-rest-api-specs` - Azure API specifications
- Custom MCP servers can be hosted for your specific needs

### Best Practices

1. **Security First**: Always implement approval workflows for sensitive operations
2. **Thread Management**: Use threads for multi-turn conversations
3. **Error Handling**: Handle network failures and approval denials gracefully
4. **Logging**: Log all function calls and approvals for audit purposes
5. **Agent Names**: Use hyphens (not underscores) in agent names

⚠️ **Security Note**: Configure approval mode server-side. For production, prefer always-require approval.